<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    3.lab.1. Wisconsin Cancer Classification - Neural Networks
  </strong>
</div>

- [ ] Implementación básica de SVM con kernel lineal y rBF
- [ ] Comparación de rendimiento entre los dos modelos
- [ ] Una visualización mostrando el límite de decisión


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

from rich.table import Table
from rich.console import Console
from sklearn import svm

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

Visual configuration of Seaborn for more attractive graphics

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
sns.set_context("notebook", font_scale=1.2)

Rich console instance to display results

In [ ]:
console = Console()

## Data Loading

In [ ]:
breast_cancer = load_breast_cancer(as_frame=True)
X = breast_cancer.data
y = breast_cancer.target

target_names = breast_cancer.target_names

diagnosis_labels = breast_cancer.target.map({0: target_names[0], 1: target_names[1]})

## Preprocessing of data

Split the data with all features into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Method to escalate the data

In [ ]:
def normalizer_data(scaler, X_train, X_test):
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

## Training Methods

Method to train a regression model

In [ ]:
def train_regression_model(X_train_scaled, y_train_data, X_test_scaled):

    model = LogisticRegression()
    model.fit(X_train_scaled, y_train_data)

    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    return y_train_pred, y_test_pred

Method to train a SVM Linear

In [ ]:
def train_svm_model(kernel, X_train_scaled, y_train_data, X_test_scaled):
    svm_model = svm.SVC(kernel=kernel, C=10000)
    svm_model.fit(X_train_scaled, y_train_data)

    svm_y_train_pred = svm_model.predict(X_train_scaled)
    svm_y_test_pred = svm_model.predict(X_test_scaled)

    return svm_y_train_pred, svm_y_test_pred

## Evaluation Methods

Method to calculate metrics

In [ ]:
def calculate_metrics_for_set(y_test, y_test_pred):
    acc = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred)

    return acc, precision, recall, f1

def calculate_metrics(
    y_train_regression_model, y_train_regression_model_pred, y_test_model, y_test_model_pred
):

    acc_train, precision_train, recall_train, f1_train = calculate_metrics_for_set(
        y_train_regression_model, y_train_regression_model_pred
    )
    acc_test, precision_test, recall_test, f1_test = calculate_metrics_for_set(
        y_test_model, y_test_model_pred
    )

    return {
        "accuracy_score_train": acc_train,
        "precision_score_train": precision_train,
        "recall_score_train": recall_train,
        "f1_score_train": f1_train,
        "accuracy_score_test": acc_test,
        "precision_score_test": precision_test,
        "recall_score_test": recall_test,
        "f1_score_test": f1_test,
    }

## Visualization methods

Method to create the comparison table

In [ ]:
def create_comparison_table(title, results):

    comparison_table_normalization = Table(title=title)

    comparison_table_normalization.add_column(
        "Model", justify="center", style="green", width=15
    )
    comparison_table_normalization.add_column(
        "Normalization", justify="left", style="cyan"
    )
    comparison_table_normalization.add_column(
        "Accuracy-\nTrain", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Accuracy-\nTest", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Precision-\nTrain", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Precision-\nTest", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Recall-\nTrain", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Recall-\nTest", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "F1-Train", justify="right", style="green"
    )
    comparison_table_normalization.add_column("F1-Test", justify="right", style="green")

    for model_name, normalization_metrics in results.items():

        for normalization_type, metrics in normalization_metrics.items():
            comparison_table_normalization.add_row(
                model_name,
                normalization_type,
                f'{metrics["accuracy_score_train"]:.4f}',
                f'{metrics["accuracy_score_test"]:.4f}',
                f'{metrics["precision_score_train"]:.4f}',
                f'{metrics["precision_score_test"]:.4f}',
                f'{metrics["recall_score_train"]:.4f}',
                f'{metrics["recall_score_test"]:.4f}',
                f'{metrics["f1_score_train"]:.4f}',
                f'{metrics["f1_score_test"]:.4f}',
            )

    console.print(comparison_table_normalization)

Method to show the confusion matrix

In [ ]:
def plot_confusion_matrix(cm, title, ax):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")


def plot_confusion_matrix_comparison(y_test, rg_y_test_pred, svm_lin_y_test_pred, svm_rbf_y_test_pred):
    cm_reg = confusion_matrix(y_test, rg_y_test_pred)
    cm_svm_linear = confusion_matrix(y_test, svm_lin_y_test_pred)
    cm_svm_rbf = confusion_matrix(y_test, svm_rbf_y_test_pred)

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    plot_confusion_matrix(cm_reg, "Confusion Matrix - Logistic Regression", axes[0][0])
    plot_confusion_matrix(cm_svm_linear, "Confusion Matrix - SVM Linear", axes[0][1])
    plot_confusion_matrix(cm_svm_rbf, "Confusion Matrix - SVM RBF", axes[1][0])

    fig.delaxes(axes[1][1])
    
    plt.tight_layout()
    plt.show()

Where the metric information for each model will be stored and the types of normalization that will be tested.

In [ ]:
all_normalization_metrics = {}

best_normalization_metrics = {}

normalizations = {
    "Min-Max": MinMaxScaler(),
    "Standard": StandardScaler(),
    "Mean Normalization": Normalizer(),
}

## 1. Regression Model

In [ ]:
all_normalization_metrics["Logistic Regression"] = {}

for normalizer_name, scaler in normalizations.items():

    X_train_scaled, X_test_scaled = normalizer_data(scaler, X_train, X_test)    

    rg_y_train_pred, rg_y_test_pred = train_regression_model(X_train_scaled, y_train, X_test_scaled)
    rg_metrics = calculate_metrics(y_train, rg_y_train_pred, y_test, rg_y_test_pred)
    all_normalization_metrics["Logistic Regression"][normalizer_name] = rg_metrics

## 2. SVM Linear Model

In [ ]:
all_normalization_metrics["SVM Linear"] = {}

for normalizer_name, scaler in normalizations.items():

    X_train_scaled, X_test_scaled = normalizer_data(scaler, X_train, X_test)

    svm_lin_y_train_pred, svm_lin_y_test_pred = train_svm_model("linear", X_train_scaled, y_train, X_test_scaled)
    svm_lin_metrics = calculate_metrics(y_train, svm_lin_y_train_pred, y_test, svm_lin_y_test_pred)
    all_normalization_metrics["SVM Linear"][normalizer_name] = svm_lin_metrics

## 3. SVM RBF Model

In [ ]:
all_normalization_metrics["SVM RBF"] = {}

for normalizer_name, scaler in normalizations.items():

    X_train_scaled, X_test_scaled = normalizer_data(scaler, X_train, X_test)
    
    svm_rbf_y_train_pred, svm_rbf_y_test_pred = train_svm_model("rbf", X_train_scaled, y_train, X_test_scaled)
    svm_rbf_metrics = calculate_metrics(y_train, svm_rbf_y_train_pred, y_test, svm_rbf_y_test_pred)
    all_normalization_metrics["SVM RBF"][normalizer_name] = svm_rbf_metrics

## 4. Comparison table of models with diferent normalization methods

In [ ]:
create_comparison_table(
    "Comparison of Logistic Regression, SVM Linear, and SVM RBF with Different Normalization Methods",
    all_normalization_metrics,
)

## 5. Comparative table of models with their best normalization

In [ ]:
best_normalization_metrics["Logistic Regression"] = {}
best_normalization_metrics["SVM Linear"] = {}
best_normalization_metrics["SVM RBF"] = {}

In [ ]:
rg_best_normalization = "Standard"
X_train_scaled_rg, X_test_scaled_rg = normalizer_data(normalizations[rg_best_normalization], X_train, X_test)

rg_y_train_pred, rg_y_test_pred = train_regression_model(X_train_scaled_rg, y_train, X_test_scaled_rg)
rg_metrics = calculate_metrics(y_train, rg_y_train_pred, y_test, rg_y_test_pred)

best_normalization_metrics["Logistic Regression"][rg_best_normalization] = rg_metrics

In [ ]:
svm_best_normalization = "Mean Normalization"

X_train_scaled_svm, X_test_scaled_svm = normalizer_data(scaler, X_train, X_test)

svm_lin_y_train_pred, svm_lin_y_test_pred = train_svm_model("linear", X_train_scaled_svm, y_train, X_test_scaled_svm)
svm_lin_metrics = calculate_metrics(y_train, svm_lin_y_train_pred, y_test, svm_lin_y_test_pred)
best_normalization_metrics["SVM Linear"][svm_best_normalization] = svm_lin_metrics

svm_rbf_y_train_pred, svm_rbf_y_test_pred = train_svm_model("rbf", X_train_scaled_svm, y_train, X_test_scaled_svm)
svm_rbf_metrics = calculate_metrics(y_train, svm_rbf_y_train_pred, y_test, svm_rbf_y_test_pred)
best_normalization_metrics["SVM RBF"][svm_best_normalization] = svm_rbf_metrics

In [ ]:
create_comparison_table(
    "Comparison of Logistic Regression, SVM Linear, and SVM RBF with Best Normalization Methods",
    best_normalization_metrics,
)